## Telco Customer Churn

#### In my project, I will explore the Telco Customer Churn dataset, emphasizing essential preprocessing techniques that prepare the data for predictive modeling. This dataset contains a range of customer attributes and account details that are crucial for understanding customer churn behavior.
#### By following this structured approach, I aim to enhance the quality of my analysis and improve the accuracy of my predictions regarding customer churn.

Load the necessary packages

In [47]:
# Import necessary libraries
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

Loading the Telco Customer dataset

In [48]:
filepath = r"C:\Users\eboat\OneDrive\Documents\DATA SCIENCE ASSIGNMENT\Telco Customer Churn Database\WA_Fn-UseC_-Telco-Customer-Churn.csv"
telco_df = pd.read_csv(filepath)
telco_df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


Start by reviewing the data info.

In [49]:
telco_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


Apply the describe function to the data

In [50]:
telco_df.describe()

,SeniorCitizen,tenure,MonthlyCharges
count,7043.000000,7043.000000,7043.000000
mean,0.162147,32.371149,64.761692
std,0.368612,24.559481,30.090047
min,0.000000,0.000000,18.250000
25%,0.000000,9.000000,35.500000
50%,0.000000,29.000000,70.350000
75%,0.000000,55.000000,89.850000
max,1.000000,72.000000,118.750000


Checking how many missing values are in our dataset

In [51]:
print(telco_df.isna().sum().sort_values())

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64


To enhance my analysis, I will implement the following steps:

In [52]:
# Handle TotalCharges errors by converting to numeric and filling non-convertible values with 0
telco_df['TotalCharges'] = pd.to_numeric(telco_df['TotalCharges'], errors='coerce').fillna(0)

# Drop the customerID column as it is not a useful feature for modeling
telco_df = telco_df.drop(columns=['customerID'])
print(telco_df.columns)

Index(['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure',
       'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
       'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV',
       'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod',
       'MonthlyCharges', 'TotalCharges', 'Churn'],
      dtype='object')


1. **Creation of Dummy Variables**: I will generate dummy variables for categorical features to transform qualitative data into a quantitative format that can be easily used in my models.

In [53]:
# To begin with, I first, map binary Yes/No columns to 1/0
binary_cols = ['Churn', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling', 'gender']
for col in binary_cols:
    if col == 'gender':
        telco_df[col] = telco_df[col].map({'Male': 1, 'Female': 0})
    else:
        telco_df[col] = telco_df[col].map({'Yes': 1, 'No': 0})

# Secondly, I will create dummy variables for the multi-category columns
multi_categorical_cols = [
    'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 
    'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 
    'Contract', 'PaymentMethod'
]
# Create dummy variables for multi-category columns 
telco_dummies = pd.get_dummies(telco_df, columns=multi_categorical_cols, drop_first=True)

2. **Data Splitting**: I will split the dataset into training and testing subsets. This step is crucial for validating the effectiveness of my predictive models and ensuring they generalize well to unseen data.

In [54]:
# Split the data into training and testing sets, ensuring that the target variable is not included in the feature set during the split.
# And also prevent data leakage by separating features and target variable before splitting the data.
X = telco_dummies.drop('Churn', axis=1)
y = telco_dummies['Churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

3. **Standardization of Numeric Features**: Finally, to ensure consistent scaling across the dataset, I will standardize the numeric features. This step is vital for improving the model's performance, especially for algorithms sensitive to the scale of input features.

In [55]:
# Scale the numerical features using StandardScaler to ensure that they are on the same scale for modeling. 
scaler = StandardScaler()
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

**Running a final check**

In [56]:
# Final Check 
print("Pre-processing Complete.")
print(f"Training shape: {X_train.shape}")
print(f"Testing shape: {X_test.shape}")
print(X_train.head())

Pre-processing Complete.
Training shape: (5634, 30)
Testing shape: (1409, 30)
      gender  SeniorCitizen  Partner  Dependents    tenure  PhoneService  \
3738       1              0        0           0  0.102371             0   
3151       1              0        1           1 -0.711743             1   
4860       1              0        1           1 -0.793155             0   
3867       0              0        1           0 -0.263980             1   
3810       1              0        1           1 -1.281624             1   

      PaperlessBilling  MonthlyCharges  TotalCharges  \
3738                 0       -0.521976     -0.262257   
3151                 0        0.337478     -0.503635   
4860                 0       -0.809013     -0.749883   
3867                 1        0.284384     -0.172722   
3810                 0       -0.676279     -0.989374   

      MultipleLines_No phone service  ...  TechSupport_Yes  \
3738                            True  ...            False   
3151